In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

print("Base directory:", BASE_DIR)
print("Raw data path:", DATA_RAW)
print("Processed data path:", DATA_PROCESSED)

Base directory: d:\projects\customer-intelligence-platform
Raw data path: d:\projects\customer-intelligence-platform\data\raw
Processed data path: d:\projects\customer-intelligence-platform\data\processed


In [3]:
import kagglehub
import shutil
from pathlib import Path

# Project paths
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = BASE_DIR / "data" / "raw"

DATA_RAW.mkdir(parents=True, exist_ok=True)

# Download Olist dataset from Kaggle
dataset_path = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))

print("Downloaded dataset path:", dataset_path)
print("Raw data folder:", DATA_RAW)

100%|██████████| 42.6M/42.6M [00:03<00:00, 11.9MB/s]

Extracting files...


Downloaded dataset path: C:\Users\silkroadnb\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2
Raw data folder: d:\projects\customer-intelligence-platform\data\raw


In [4]:
csv_files = list(dataset_path.glob("*.csv"))

for file in csv_files:
    destination = DATA_RAW / file.name
    shutil.copy(file, destination)

print(f"Copied {len(csv_files)} CSV files to {DATA_RAW}")

for file in sorted(DATA_RAW.glob("*.csv")):
    print(file.name)

Copied 9 CSV files to d:\projects\customer-intelligence-platform\data\raw
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [5]:
orders = pd.read_csv(DATA_RAW / "olist_orders_dataset.csv")
items = pd.read_csv(DATA_RAW / "olist_order_items_dataset.csv")
customers = pd.read_csv(DATA_RAW / "olist_customers_dataset.csv")
products = pd.read_csv(DATA_RAW / "olist_products_dataset.csv")
payments = pd.read_csv(DATA_RAW / "olist_order_payments_dataset.csv")

tables = {
    "orders": orders,
    "items": items,
    "customers": customers,
    "products": products,
    "payments": payments,
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")

orders: (99441, 8)
items: (112650, 7)
customers: (99441, 5)
products: (32951, 9)
payments: (103886, 5)


## 2. Initial Data Quality Checks

Before merging the tables, I check the shape, missing values, duplicate rows, and key business columns in each dataset. This step helps identify data quality issues and prevents incorrect joins or misleading KPI calculations.

In [11]:
for name, df in tables.items():
    print("=" * 80)
    print(f"Table: {name}")
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    display(df.head())

Table: orders
Shape: (99441, 8)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


Table: items
Shape: (112650, 7)

Columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


Table: customers
Shape: (99441, 5)

Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


Table: products
Shape: (32951, 9)

Columns:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


Table: payments
Shape: (103886, 5)

Columns:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [12]:
def missing_summary(df):
    summary = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean() * 100
    })
    return summary.sort_values("missing_percent", ascending=False)

for name, df in tables.items():
    print("=" * 80)
    print(f"Missing values in {name}")
    display(missing_summary(df).head(10))

Missing values in orders


,missing_count,missing_percent
order_delivered_customer_date,2965,2.981668
order_delivered_carrier_date,1783,1.793023
order_approved_at,160,0.160899
order_id,0,0.000000
order_purchase_timestamp,0,0.000000
order_status,0,0.000000
customer_id,0,0.000000
order_estimated_delivery_date,0,0.000000


Missing values in items


,missing_count,missing_percent
order_id,0,0.0
order_item_id,0,0.0
product_id,0,0.0
seller_id,0,0.0
shipping_limit_date,0,0.0
price,0,0.0
freight_value,0,0.0


Missing values in customers


,missing_count,missing_percent
customer_id,0,0.0
customer_unique_id,0,0.0
customer_zip_code_prefix,0,0.0
customer_city,0,0.0
customer_state,0,0.0


Missing values in products


,missing_count,missing_percent
product_category_name,610,1.851234
product_description_lenght,610,1.851234
product_name_lenght,610,1.851234
product_photos_qty,610,1.851234
product_weight_g,2,0.006070
product_height_cm,2,0.006070
product_length_cm,2,0.006070
product_width_cm,2,0.006070
product_id,0,0.000000


Missing values in payments


,missing_count,missing_percent
order_id,0,0.0
payment_sequential,0,0.0
payment_type,0,0.0
payment_installments,0,0.0
payment_value,0,0.0


In [13]:
duplicate_summary = []

for name, df in tables.items():
    duplicate_summary.append({
        "table": name,
        "rows": df.shape[0],
        "duplicate_rows": df.duplicated().sum(),
        "duplicate_percent": round(df.duplicated().mean() * 100, 3)
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary)
duplicate_summary_df

,table,rows,duplicate_rows,duplicate_percent
0,orders,99441,0,0.0
1,items,112650,0,0.0
2,customers,99441,0,0.0
3,products,32951,0,0.0
4,payments,103886,0,0.0


In [14]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [15]:
order_status_summary = (
    orders["order_status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .reset_index()
)

order_status_summary.columns = ["order_status", "percent"]
order_status_summary

,order_status,percent
0,delivered,97.02
1,shipped,1.11
2,canceled,0.63
3,unavailable,0.61
4,invoiced,0.32
5,processing,0.30
6,created,0.01
7,approved,0.00


Most orders are expected to be delivered. Non-delivered statuses such as canceled or unavailable should be treated carefully because they can distort revenue and delivery-time analysis.

In [16]:
key_checks = {
    "orders_order_id_unique": orders["order_id"].is_unique,
    "customers_customer_id_unique": customers["customer_id"].is_unique,
    "products_product_id_unique": products["product_id"].is_unique,
    "items_order_id_unique": items["order_id"].is_unique,
    "payments_order_id_unique": payments["order_id"].is_unique,
}

key_checks

{'orders_order_id_unique': True,
 'customers_customer_id_unique': True,
 'products_product_id_unique': True,
 'items_order_id_unique': False,
 'payments_order_id_unique': False}

The orders, customers, and products tables have unique primary keys. However, order_items and payments are one-to-many tables at the order level, so they must be aggregated carefully before creating order-level or customer-level datasets.

## 3. Build Clean Order-Level Analytical Dataset

The Olist dataset contains multiple tables with different granularities. 
The orders, customers, and products tables have mostly unique keys, while order_items and payments are one-to-many at the order level. 

To avoid duplicated revenue or payment values, I first aggregate item-level and payment-level data to the order level, then merge them with order and customer information.

In [17]:
# Convert order timestamp columns to datetime
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders[date_cols].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [18]:
items_enriched = items.merge(
    products[[
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]],
    on="product_id",
    how="left"
)

print("items shape:", items.shape)
print("items_enriched shape:", items_enriched.shape)

items_enriched.head()

items shape: (112650, 7)
items_enriched shape: (112650, 12)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,650.0,28.0,9.0,14.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,30000.0,50.0,30.0,40.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,3050.0,33.0,13.0,33.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,200.0,16.0,10.0,15.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,3750.0,35.0,40.0,30.0


In [19]:
def get_primary_category(group):
    """
    Return the product category with the highest item revenue within an order.
    If category is missing, return NaN.
    """
    category_revenue = (
        group.dropna(subset=["product_category_name"])
        .groupby("product_category_name")["price"]
        .sum()
        .sort_values(ascending=False)
    )
    
    if category_revenue.empty:
        return np.nan
    
    return category_revenue.index[0]

In [20]:
item_order_agg = (
    items_enriched
    .groupby("order_id")
    .agg(
        product_revenue=("price", "sum"),
        shipping_revenue=("freight_value", "sum"),
        total_items=("order_item_id", "count"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
        unique_categories=("product_category_name", "nunique"),
        avg_item_price=("price", "mean"),
        max_item_price=("price", "max"),
        avg_freight_value=("freight_value", "mean")
    )
    .reset_index()
)

primary_categories = (
    items_enriched
    .groupby("order_id")
    .apply(get_primary_category)
    .reset_index(name="primary_product_category")
)

item_order_agg = item_order_agg.merge(
    primary_categories,
    on="order_id",
    how="left"
)

item_order_agg["total_order_value"] = (
    item_order_agg["product_revenue"] + item_order_agg["shipping_revenue"]
)

item_order_agg["freight_ratio"] = (
    item_order_agg["shipping_revenue"] / item_order_agg["product_revenue"].replace(0, np.nan)
)

item_order_agg.head()

,order_id,product_revenue,shipping_revenue,total_items,unique_products,unique_sellers,unique_categories,avg_item_price,max_item_price,avg_freight_value,primary_product_category,total_order_value,freight_ratio
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,1,1,1,58.90,58.90,13.29,cool_stuff,72.19,0.225637
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,1,1,1,239.90,239.90,19.93,pet_shop,259.83,0.083076
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,1,1,1,199.00,199.00,17.87,moveis_decoracao,216.87,0.089799
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,1,1,1,12.99,12.99,12.79,perfumaria,25.78,0.984604
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,1,1,1,199.90,199.90,18.14,ferramentas_jardim,218.04,0.090745


In [21]:
def get_main_payment_type(group):
    """
    Return the payment type with the highest total payment value for an order.
    """
    payment_by_type = (
        group.groupby("payment_type")["payment_value"]
        .sum()
        .sort_values(ascending=False)
    )
    
    if payment_by_type.empty:
        return np.nan
    
    return payment_by_type.index[0]


payment_order_agg = (
    payments
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        number_of_payment_records=("payment_sequential", "count"),
        number_of_payment_types=("payment_type", "nunique")
    )
    .reset_index()
)

main_payment_type = (
    payments
    .groupby("order_id")
    .apply(get_main_payment_type)
    .reset_index(name="main_payment_type")
)

payment_order_agg = payment_order_agg.merge(
    main_payment_type,
    on="order_id",
    how="left"
)

payment_order_agg.head()

,order_id,total_payment_value,payment_installments,number_of_payment_records,number_of_payment_types,main_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1,1,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1,1,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1,1,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,1,1,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1,1,credit_card


In [22]:
order_base = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

print("orders shape:", orders.shape)
print("order_base shape:", order_base.shape)

order_base.head()

orders shape: (99441, 8)
order_base shape: (99441, 12)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [23]:
order_level = (
    order_base
    .merge(item_order_agg, on="order_id", how="left")
    .merge(payment_order_agg, on="order_id", how="left")
)

print("Final order_level shape:", order_level.shape)
order_level.head()

Final order_level shape: (99441, 29)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_revenue,shipping_revenue,total_items,unique_products,unique_sellers,unique_categories,avg_item_price,max_item_price,avg_freight_value,primary_product_category,total_order_value,freight_ratio,total_payment_value,payment_installments,number_of_payment_records,number_of_payment_types,main_payment_type
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,29.99,8.72,1.0,1.0,1.0,1.0,29.99,29.99,8.72,utilidades_domesticas,38.71,0.290764,38.71,1.0,3.0,2.0,voucher
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,118.70,22.76,1.0,1.0,1.0,1.0,118.70,118.70,22.76,perfumaria,141.46,0.191744,141.46,1.0,1.0,1.0,boleto
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,159.90,19.22,1.0,1.0,1.0,1.0,159.90,159.90,19.22,automotivo,179.12,0.120200,179.12,3.0,1.0,1.0,credit_card
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,45.00,27.20,1.0,1.0,1.0,1.0,45.00,45.00,27.20,pet_shop,72.20,0.604444,72.20,1.0,1.0,1.0,credit_card
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,19.90,8.72,1.0,1.0,1.0,1.0,19.90,19.90,8.72,papelaria,28.62,0.438191,28.62,1.0,1.0,1.0,credit_card


In [24]:
order_level["order_date"] = order_level["order_purchase_timestamp"].dt.date
order_level["order_month"] = order_level["order_purchase_timestamp"].dt.to_period("M").astype(str)
order_level["order_year"] = order_level["order_purchase_timestamp"].dt.year
order_level["order_quarter"] = order_level["order_purchase_timestamp"].dt.quarter
order_level["order_day_of_week"] = order_level["order_purchase_timestamp"].dt.day_name()

order_level["delivery_days"] = (
    order_level["order_delivered_customer_date"] - order_level["order_purchase_timestamp"]
).dt.days

order_level["estimated_delivery_days"] = (
    order_level["order_estimated_delivery_date"] - order_level["order_purchase_timestamp"]
).dt.days

order_level["delivery_delay"] = (
    order_level["order_delivered_customer_date"] - order_level["order_estimated_delivery_date"]
).dt.days

order_level["is_late_delivery"] = np.where(
    order_level["delivery_delay"] > 0,
    1,
    0
)

order_level["is_delivered"] = np.where(
    order_level["order_status"] == "delivered",
    1,
    0
)

order_level[[
    "order_id",
    "order_status",
    "order_month",
    "product_revenue",
    "shipping_revenue",
    "total_order_value",
    "total_payment_value",
    "delivery_days",
    "delivery_delay",
    "is_late_delivery"
]].head()

,order_id,order_status,order_month,product_revenue,shipping_revenue,total_order_value,total_payment_value,delivery_days,delivery_delay,is_late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10,29.99,8.72,38.71,38.71,8.0,-8.0,0
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07,118.70,22.76,141.46,141.46,13.0,-6.0,0
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08,159.90,19.22,179.12,179.12,9.0,-18.0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11,45.00,27.20,72.20,72.20,13.0,-13.0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02,19.90,8.72,28.62,28.62,2.0,-10.0,0


### Revenue Definition

In this project, I separate product revenue from shipping revenue:

- `product_revenue`: sum of product item prices within an order
- `shipping_revenue`: sum of freight values within an order
- `total_order_value`: product revenue + shipping revenue
- `total_payment_value`: total amount paid by the customer

For most KPI and customer value analyses, I use `product_revenue` as the main revenue metric because freight is a logistics charge and may not represent product sales revenue. However, `total_order_value` and `freight_ratio` are kept for operational and pricing analysis.

In [25]:
sanity_checks = {
    "orders_original_rows": orders.shape[0],
    "order_level_rows": order_level.shape[0],
    "unique_orders_original": orders["order_id"].nunique(),
    "unique_orders_order_level": order_level["order_id"].nunique(),
    "total_item_revenue_original": round(items["price"].sum(), 2),
    "total_product_revenue_order_level": round(order_level["product_revenue"].sum(), 2),
    "total_freight_original": round(items["freight_value"].sum(), 2),
    "total_shipping_revenue_order_level": round(order_level["shipping_revenue"].sum(), 2),
    "total_payment_original": round(payments["payment_value"].sum(), 2),
    "total_payment_order_level": round(order_level["total_payment_value"].sum(), 2),
}

sanity_checks

{'orders_original_rows': 99441,
 'order_level_rows': 99441,
 'unique_orders_original': 99441,
 'unique_orders_order_level': 99441,
 'total_item_revenue_original': np.float64(13591643.7),
 'total_product_revenue_order_level': np.float64(13591643.7),
 'total_freight_original': np.float64(2251909.54),
 'total_shipping_revenue_order_level': np.float64(2251909.54),
 'total_payment_original': np.float64(16008872.12),
 'total_payment_order_level': np.float64(16008872.12)}

In [26]:
processed_file_path = DATA_PROCESSED / "order_level_dataset.csv"

order_level.to_csv(processed_file_path, index=False)

print(f"Saved processed order-level dataset to: {processed_file_path}")
print("Shape:", order_level.shape)

Saved processed order-level dataset to: d:\projects\customer-intelligence-platform\data\processed\order_level_dataset.csv
Shape: (99441, 39)


### Data Preparation Summary

I created an order-level analytical dataset by aggregating item-level and payment-level tables before merging. This prevents duplicated revenue and payment values caused by one-to-many joins. The resulting dataset preserves one row per order and includes customer information, revenue metrics, delivery features, product category information, and payment summaries.